# <u> NOTEBOOK PERMETTANT DE REPONDRE AUX DEMANDES D'ANNABELLE </U>

### <u> Import des librairies utiles pour ce projet </u>

In [ ]:
# Outils

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import plotly.subplots as sp
import plotly.graph_objects as go
import statsmodels.api as sm

In [ ]:
# Palette de couleurs

px.colors.qualitative.LibrairieContrast = [
    "#2f3a4a",  
    "#7f5083",  
    "#e95c6d",  
    "#ffa600",  
    "#4c9f70", 
    "#3b7dd8", 
    "#b95383"   
]

px.colors.qualitative.Librairie3 = [
    "#e95c6d",
    "#2f3a4a",
    "#ffa600",
]

palette = {
    "0": "#e95c6d",
    "1": "#2f3a4a",
    "2": "#ffa600"
}

### <u> Chargement du dataset selon les csv remis lors de la mission </u>

In [ ]:
df_lapage=pd.read_csv(r'..\..\projet9_Lapage\data\processed\df_lapage.csv',sep=';', encoding='utf-8')

In [ ]:
print('Les données minimales par colonnes du dataframe sont les suivantes :')
print(df_lapage.min())

print('\n ----------------------------------------------------------')
print('Les données maximales par colonnes du dataframe sont les suivantes :')
print(df_lapage.max())

print('\n ----------------------------------------------------------')
print('Le type de chaque colonne du dataframe est :')
print(df_lapage.dtypes)

In [ ]:
# Conversion de la colonne date en datetime
df_lapage['date'] = pd.to_datetime(df_lapage['date'])

## <u> 1. Le chiffre d'affaires </U>

#### Chiffre d’affaires avec la moyenne mobile : jour, semaine, mois

In [ ]:
# Agregation en jour pour ensuite effectuer la moyenne glissante en semaine 
df_ca = df_lapage.groupby(pd.Grouper(key='date', freq='D'))['price'].sum().to_frame().reset_index()

In [ ]:
# Création de la colonne indiquant une moyenne glissante sur une semaine
df_ca['Moyenne glissante (semaine)']=df_ca['price'].rolling(window=7).mean().round(2)

In [ ]:
# Création de la colonne indiquant une moyenne glissante sur un mois
df_ca['Moyenne glissante (mois)']=df_ca['price'].rolling(window=30).mean().round(2)

In [ ]:
# Vérification de la création des colonnes de moyennes glissantes
df_ca.head(35)

In [ ]:
# Grahique concernant la saisonnalité du CA. 
# Pas de réelle saisonnalité : pics principalement sur 4 jours : entre jeudi et dimanche
# Interprétation limitée et tendance à une certaine stabilité - saisonnalité marquée mais pas de pics

figure = px.line(
    df_ca, 
    x='date', 
    y=['price', 'Moyenne glissante (semaine)','Moyenne glissante (mois)'], 
    color_discrete_sequence=list(palette.values()),
    markers=False,
    title='Évolution du chiffre d\'affaires',
    labels={
        'date': 'Date (mois)',
        'value': 'Chiffre d’affaires (€)',
        'variable': 'Moyennes glissantes'
    }
)

figure.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure.update_traces(
    hovertemplate='%{y:.0f}€'
)

figure.show()

In [ ]:
# Quanti - quanti



#### Chiffre d’affaires par catégorie + évolution dans le temps

In [ ]:
# Création d'un df contenant les données agregées par mois
df_lapage_mois = df_lapage.groupby([pd.Grouper(key='date', freq='MS'), 'categ'])['price'].sum().reset_index()
df_lapage_mois['date AAAA-MM'] = df_lapage_mois['date'].dt.strftime('%Y-%m')

In [ ]:
# Vérification des catégories uniques présentes dans la colonne categ
categories =df_lapage_mois.categ.unique()
print('Les catégories uniques du df sont les suivantes :', categories)

In [ ]:
# Affichage du chiffre d'affaire par catégorie sur l'intégralité de la période du dataset
for categ in df_lapage_mois.categ.unique():
    ca = df_lapage_mois[df_lapage_mois['categ'] == categ]['price'].sum()
    print (f'Le chiffre d\'affaire total de la catégorie {categ} est de {ca:,.0f} euros'.replace(',',' '))

In [ ]:
figure2 = px.line(
    df_lapage_mois, 
    x='date', 
    y='price', 
    color='categ',
    color_discrete_sequence=list(palette.values()),
    markers=False,
    title='Évolution mensuelle du chiffre d’affaires par catégorie',
    labels={
        'date': 'Date (mois)',
        'price': 'Chiffre d’affaires (€)',
        'categ': 'Catégories'
    }
)

figure2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure2.update_traces(
    hovertemplate='%{y:.0f}€'
    )


figure2.show()

## <u> 2. Les clients </U>

#### Nombre de clients par mois + évolution dans le temps

In [ ]:
#Affichage du nombre de clients au total sur la période du dataset
print(f'Sur l\'intégralité de la période de référence du dataset, il y a eu un total de {df_lapage['client_id'].nunique()} clients distincts.')

In [ ]:
#Affichage du nombre de clients uniques par mois
clients_par_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].nunique().reset_index(name='Nombre de clients'))
clients_par_mois.head()

In [ ]:
figure3=px.line(
    clients_par_mois,
    x='date',
    y='Nombre de clients',
    color_discrete_sequence=list(palette.values()),
    markers=True,
    title='Évolution mensuelle du nombre de clients (uniques)',
    labels={
        'date': 'Date (mois)'
    }
)

figure3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure3.show()

In [ ]:
#Affichage du nombre de clients par mois
clients_total_par_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].count().reset_index(name='Nombre de clients'))
clients_total_par_mois.head()

In [ ]:
figure4=px.line(
    clients_total_par_mois,
    x='date',
    y='Nombre de clients',
    color_discrete_sequence=list(palette.values()),
    markers=True,
    title='Évolution mensuelle du nombre de clients',
    labels={
        'date': 'Date (mois)'
    }
)

figure4.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure4.show()

## <u> 3. Les transactions </U>

#### Nombre de transactions + évolution dans le temps

In [ ]:
#Affichage du nombre de transactions totales sur la période du dataset
print(f'Sur l\'intégralité de la période de référence du dataset, il y a eu un total de {df_lapage['session_id'].count()} transactions.')

In [ ]:
#Affichage du nombre de transactions par semaine

# Création d'un dataframe servant de base quotidienne
df_quotidien = df_lapage.groupby(pd.Grouper(key='date', freq='D'))['session_id'].count().reset_index(name='Transactions_jour')

# Puis création des colonnes de tendances 
df_quotidien['Tendance_Semaine'] = df_quotidien['Transactions_jour'].rolling(window=7).mean()
df_quotidien['Tendance_Mois'] = df_quotidien['Transactions_jour'].rolling(window=30, center=True).mean()
df_quotidien['Tendance_Trimestre'] = df_quotidien['Transactions_jour'].rolling(window=90, center=True).mean()
# center=True calcule la moyenne en prenant les jours avant et après le point central plutôt que la moyenne des X jours précédents.
### Prendre en compte le fait que la période commence le 01/03/21 donc le premier trimestre n'est pas significatif
### Le dernier trimestre de la période du dataset ne sera pas significatif non plus car il ne couvre que janvier et février 2023

In [ ]:
figure5=px.line(
    df_quotidien,
    x='date',
    y=['Tendance_Semaine', 'Tendance_Mois', 'Tendance_Trimestre'],    
    color_discrete_sequence=list(palette.values()),
    markers=False,
    title='Évolution du nombre de transactions par semaine, mois et trimestre',
    labels={
        'date': 'Date (mois)'
    }
)

figure5.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)


## <u> 4. Les produits </U>

#### Nombre de produits vendus + évolution dans le temps

In [ ]:
# tous les produits sur toute la période, puis par jour, mois, par semaine, par trimestre
#Affichage du nombre de produits vendus par jour
df_prod = (df_lapage.groupby(pd.Grouper(key='date', freq='D'))['id_prod'].nunique().reset_index(name='Tendance_Prod_Jour'))
df_prod

In [ ]:
#Affichage du nombre de produits vendus par semaine
df_prod['Tendance_Prod_Semaine'] = df_prod['Tendance_Prod_Jour'].rolling(window=7, center=True).mean()

In [ ]:
#Affichage du nombre de produits vendus par semaine
df_prod['Tendance_Prod_Mois'] = df_prod['Tendance_Prod_Jour'].rolling(window=30, center=True).mean()

In [ ]:
#Affichage du nombre de produits vendus par trimestre
df_prod['Tendance_Prod_Trimestre'] = df_prod['Tendance_Prod_Jour'].rolling(window=90, center=True).mean()

In [ ]:
figure6=px.line(
    df_prod,
    x='date',
    y=['Tendance_Prod_Semaine','Tendance_Prod_Mois','Tendance_Prod_Trimestre'],    
    color_discrete_sequence=list(palette.values()),
    markers=False,
    title='Évolution du nombre de produits uniques vendus par semaine, mois et trimestre',
    labels={
        'date': 'Date (mois)'
    }
)

figure6.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure6.update_traces(
    hovertemplate='%{y:.0f}€'
    )

figure6.show()

### <u> Top, Flop, la répartition par catégorie</u>

### <u> ***Le Top 10 des produits*** </u> 

In [ ]:
# Top 10 par catégorie

In [ ]:
# Top 10 global toutes catégories confondues
top10_data = df_lapage['id_prod'].value_counts().head(10).reset_index()
top10_data.columns = ['id_prod', 'nombre_de_ventes']
top10_data

In [ ]:
# reprise le 29 03 26
# Top 10 - chaque ligne comprend un 1 produit / une date / un id de session / un client (et ses infos) / un prix / la catégorie 
top_par_categ = df_lapage.groupby(['categ', 'id_prod']).size().reset_index(name='ventes')
top10_par_categ = top_par_categ.sort_values(['categ', 'ventes'], ascending=[True, False]).groupby('categ').head(10)
top_par_categ

In [ ]:
# Nombre de ventes par produit et par catégorie + top 10
top_par_categ = df_lapage.groupby(['categ', 'id_prod']).size().reset_index(name='ventes')
top10_par_categ = top_par_categ.sort_values(['categ', 'ventes'], ascending=[True, False]).groupby('categ').head(10)

In [ ]:
top10_par_categ['categ'] = top10_par_categ['categ'].astype(str)
fig2 = px.bar(top10_par_categ, 
        x='ventes', 
        y='id_prod', 
        color='categ', 
        color_discrete_map=palette,
        facet_col='categ',
        facet_col_wrap=3,  
        orientation='h',
        labels={'id_prod': 'Produit', 'ventes': 'Nombre de ventes','categ': 'Catégorie'})

fig2.update_layout(title_text='Top 10 des ventes par Catégorie', title_x=0.5)
fig2.update_yaxes(matches=None)
fig2.show()

### <u> ***Le Flop 10*** </u> 

In [ ]:
# Nombre de ventes par produit et par catégorie + flop 10
flop_par_categ = df_lapage.groupby(['categ', 'id_prod']).size().reset_index(name='ventes')
flop10_par_categ = flop_par_categ.sort_values(['categ', 'ventes'], ascending=[True, True]).groupby('categ').head(10)

In [ ]:
flop10_par_categ['categ'] = flop10_par_categ['categ'].astype(str)
fig3 = px.bar(flop10_par_categ, 
        x='ventes', 
        y='id_prod', 
        color='categ', 
        color_discrete_map=palette,
        facet_col='categ',
        facet_col_wrap=3,  
        orientation='h',
        labels={'id_prod': 'Produit', 'ventes': 'Nombre de ventes','categ': 'Catégorie'})

fig3.update_layout(title_text='Flop 10 des ventes par Catégorie', title_x=0.5)
fig3.update_yaxes(matches=None)
fig3.show()

In [ ]:
# Nombre de références vendues une seule fois
counts = df_lapage['id_prod'].value_counts()
produits_uniques = counts[counts == 1].reset_index()

print(f"Le nombre de livres vendus une seule fois : {len(produits_uniques)} références")

In [ ]:
# Liste des "flops" avec la catégorie
flops = produits_uniques.merge(df_lapage[['id_prod', 'categ']].drop_duplicates(), on='id_prod', how='left')

categ_flops = flops['categ'].value_counts().reset_index()
categ_flops

Nous observons que 18 produits n'ont été vendus qu'une seule fois sur la période. La majorité de ces références appartiennent à la catégorie 0, ce qui pourrait indiquer un besoin de renouvellement du catalogue sur ce segment.

## <u> 5. Le profil de nos clients </U>

### <u> SEGMENTATION DES CLIENTS : BtoB ET BtoC </u>

#### ***Il existe plusieurs moyens pour identifier les B to B :*** 

1. Classification selon le montant du CA total :    

In [ ]:
# Nous identifions 4 clients ayant généré plusieurs centaines de milliers d'euros de CA
ca_par_client = df_lapage.groupby(['client_id'])['price'].sum().reset_index().sort_values('price', ascending=False)
ca_par_client.head(10)
# clients = ['c_1609','c_4958','c_6714','c_3454']

In [ ]:
# Création d'une colonne dans le df, précisant s'il s'agit d'un client B2B ou B2C selon le tri ci-dessus
clients = ['c_1609','c_4958','c_6714','c_3454']
df_lapage['type_client']=df_lapage['client_id'].apply(lambda x: 'B to B' if x in clients else 'B to C')
df_lapage.head(20)

# Création d'un df b2b 
df_b2b = df_lapage[df_lapage['client_id'].isin(clients)]

# Mise à jour du df principal excluant ces 4 clients
df_b2c = df_lapage[df_lapage['type_client'] == 'B to C'].copy()
df_b2c

In [ ]:
# répartition dezs clients b2c par catégorie
client_categorie=df_b2c.groupby('categ')['id_prod'].nunique().reset_index()
client_categorie.columns = ['categ', 'nombre_produits_uniques']

In [ ]:
palette = list(palette.values())

fig_clients = px.pie(
    client_categorie, 
    values='nombre_produits_uniques', 
    names='categ',
    title='Répartition des produits vendus (B2C) par catégorie',
    color_discrete_sequence=palette
)
# fig_clients.update_layout(title_text='Répartition des produits vendus (B2C) par catégorie', title_x=0.5)
fig_clients.show()

2. Classification selon le nombre d'exemplaires achetés, pour chaque produit :

In [ ]:
df_b2b

In [ ]:
figureb2b = px.histogram(
    df_b2b,
    x="client_id",
    color="categ",
    barmode="group",
    nbins=20,
    color_discrete_sequence=palette,
    title="Histogramme représentant le nombre d'achats par clients B to B, par catégories"
)

figureb2b.show()

In [ ]:
var_a = df_b2b.groupby(['client_id','categ'])['price'].sum().reset_index()
df_b2b

In [ ]:
figureb2b_ca = px.histogram(
    df_b2b,
    x="client_id",
    y="price",
    barmode="group",
    nbins=20,
    color_discrete_sequence=palette,
    title="Histogramme représentant le montant du CA par clients B to B, par catégories"
)

figureb2b_ca.show()

 ***Le fait de trier les B2B sur la quantité de livres achetés n'est pas cohérent car une même personne peut acheter plusieurs exemplaires d'un même ouvrage*** 
 ***(sortie de livres, achat pour offrir ...) : 425 clients***

### <u> Utilisation du df_b2c pour identifier le profil des clients </u> 

In [ ]:
# classement de l'Age des clients par catégorie
df_b2c.sort_values(by=['categ', 'birth'], ascending=[True, True]).reset_index(drop=True)

In [ ]:
df_b2c.info()
df_b2c.describe()

In [ ]:
# Création de la colonne age, correspondant à la différence entre la date du jour et l'année de 'birth' 
# Pour que la bonne catégorie d'âge soit retenue, il faut prendre la date d'achat comme date de référence

df_b2c['date'] = pd.to_datetime(df_b2c['date'])
df_b2c['birth'] = pd.to_datetime(df_b2c['birth'], format='%Y')
df_b2c['age'] = (df_b2c['date'] - df_b2c['birth']).dt.days // 365

In [ ]:
df_b2c

In [ ]:
figure6 = px.histogram(
    df_b2c,
    x="age",
    color="categ",
    barmode="group",
    nbins=20,
    color_discrete_sequence=palette,
    title="Histogramme empilé du CA par âges et catégorie"
)

figure6.show()


In [ ]:
top10_ca=df_b2c.groupby('client_id')['price'].sum().reset_index(name='CA_total')
top10_ca.columns = ['client_id', 'CA_total']
top10_ca=top10_ca.sort_values(by='CA_total',ascending=False).head(10)

In [ ]:
# Graphique du Top 10 des clients en terme de CA généré
fig_top10_ca = px.bar(
    top10_ca, 
    x= 'client_id', 
    y='CA_total',
    title='Top 10 des clients B2C par Chiffre d\'Affaires',
    labels={'client_id': 'Identifiant Client', 'CA_total': 'Chiffre d\'Affaires Total (€)'},
    color='client_id',
    color_discrete_sequence=palette
)
fig_top10_ca.update_layout(
    yaxis={'categoryorder':'total ascending'},
    showlegend=False                           
)
fig_top10_ca.show()


In [ ]:
# Flop 20 des clients en terme de CA généré
flop10_ca=df_b2c.groupby('client_id')['price'].sum().reset_index(name='CA_total')
flop10_ca.columns = ['client_id', 'CA_total']
flop10_ca=flop10_ca.sort_values(by='CA_total',ascending=True).head(10)

In [ ]:
# Graphique du Flop 10 des clients en terme de CA généré
fig_flop10_ca = px.bar(
    flop10_ca, 
    x= 'client_id', 
    y='CA_total',
    title='Flop 10 des clients B2C par Chiffre d\'Affaires',
    labels={'client_id': 'Identifiant Client', 'CA_total': 'Chiffre d\'Affaires Total (€)'},
    color='client_id',
    color_discrete_sequence=palette
)
fig_top10_ca.update_layout(
    showlegend=False                           
)
fig_flop10_ca.show()


La courbe de Lorenz permet de visualiser la concentration du CA. 
Plus la courbe est "creusée", plus le CA est concentré sur quelques gros clients.

In [ ]:
ca_b2b = df_b2b.groupby('client_id')['price'].sum().values
n = len(ca_b2b)

# 2. Trier les CA par ordre croissant (obligatoire pour Lorenz)
lorenz = np.sort(ca_b2b)

# 3. Calculer la somme cumulée des CA (ordonnées) et la part de population (abscisses)
lorenz_cum = np.cumsum(lorenz) / lorenz.sum()
lorenz_cum = np.append([0], lorenz_cum) # On commence à 0

# Part cumulée de la population (de 0 à 1)
xaxis = np.linspace(0-1/n, 1, len(lorenz_cum)) 

# 4. Calcul de l'indice de Gini
# Aire sous la courbe de Lorenz
AUC = (lorenz_cum.sum() - lorenz_cum[-1]/2 - lorenz_cum[0]/2) / n
# Aire entre la diagonale et la courbe
S = 0.5 - AUC
gini = 2 * S

# 5. Graphique
plt.figure(figsize=(8,6))

plt.plot(xaxis, lorenz_cum, color="#e95c6d", label='Courbe de Lorenz') 
plt.plot([0,1], [0,1],color="#2f3a4a", linestyle='--', label='Ligne d\'égalité parfaite')
plt.fill_between(xaxis, xaxis, lorenz_cum, color="#2f3a4a", alpha=0.2)
plt.title(f'Courbe de Lorenz - Concentration du CA B2B\nIndice de Gini : {gini:.2f}')
plt.xlabel('Part cumulée des clients')
plt.ylabel('Part cumulée du CA')
plt.legend()
plt.grid(False)
plt.show()



Un Gini de 0.22 sur les 4 gros clients signifie qu'ils sont "homogènes dans l'excès".

Ils achètent tous énormément.

Leurs chiffres d'affaires respectifs sont proches les uns des autres (par exemple, ils font tous entre 150k€ et 250k€).

Analyse : Entre "géants", il n'y a pas de grosse inégalité. Ils forment un bloc à part.
Groupe homogène : 4 partenaires majeurs aux volumes similaires.

In [ ]:
ca_b2c = df_b2c.groupby('client_id')['price'].sum().values
n = len(ca_b2c)

# 2. Trier les CA par ordre croissant (obligatoire pour Lorenz)
lorenz = np.sort(ca_b2c)

# 3. Calculer la somme cumulée des CA (ordonnées) et la part de population (abscisses)
lorenz_cum = np.cumsum(lorenz) / lorenz.sum()
lorenz_cum = np.append([0], lorenz_cum) # On commence à 0

# Part cumulée de la population (de 0 à 1)
xaxis = np.linspace(0-1/n, 1, len(lorenz_cum)) 

# 4. Calcul de l'indice de Gini
# Aire sous la courbe de Lorenz
AUC = (lorenz_cum.sum() - lorenz_cum[-1]/2 - lorenz_cum[0]/2) / n
# Aire entre la diagonale et la courbe
S = 0.5 - AUC
gini = 2 * S

# 5. Graphique
plt.figure(figsize=(8,6))

plt.plot(xaxis, lorenz_cum, color="#e95c6d", label='Courbe de Lorenz') 
plt.plot([0,1], [0,1],color="#2f3a4a", linestyle='--', label='Ligne d\'égalité parfaite')
plt.fill_between(xaxis, xaxis, lorenz_cum, color="#2f3a4a", alpha=0.2)
plt.title(f'Courbe de Lorenz - Concentration du CA B2C\nIndice de Gini : {gini:.2f}')
plt.xlabel('Part cumulée des clients')
plt.ylabel('Part cumulée du CA')
plt.legend()
plt.grid(False)
plt.show()

Un Gini de 0.40, c'est l'indice classique d'une activité commerciale saine mais contrastée.

Cela suit souvent la Loi de Pareto : environ 20% des clients font 50% à 60% du CA.

Analyse : Tu as beaucoup de "petits" clients qui achètent un livre de temps en temps, et une base de clients fidèles ("gros lecteurs") qui achètent très régulièrement. C'est ce qui "creuse" la courbe.

Clientèle hétérogène : présence de "gros lecteurs" fidèles.

In [ ]:
ca_clients = df_lapage.groupby('client_id')['price'].sum().values
n = len(ca_clients)

# 2. Trier les CA par ordre croissant (obligatoire pour Lorenz)
lorenz = np.sort(ca_clients)

# 3. Calculer la somme cumulée des CA (ordonnées) et la part de population (abscisses)
lorenz_cum = np.cumsum(lorenz) / lorenz.sum()
lorenz_cum = np.append([0], lorenz_cum) # On commence à 0

# Part cumulée de la population (de 0 à 1)
xaxis = np.linspace(0-1/n, 1, len(lorenz_cum)) 

# 4. Calcul de l'indice de Gini
# Aire sous la courbe de Lorenz
AUC = (lorenz_cum.sum() - lorenz_cum[-1]/2 - lorenz_cum[0]/2) / n
# Aire entre la diagonale et la courbe
S = 0.5 - AUC
gini = 2 * S

# 5. Graphique
plt.figure(figsize=(8,6))

plt.plot(xaxis, lorenz_cum, color="#e95c6d", label='Courbe de Lorenz') 
plt.plot([0,1], [0,1],color="#2f3a4a", linestyle='--', label='Ligne d\'égalité parfaite')
plt.fill_between(xaxis, xaxis, lorenz_cum, color="#2f3a4a", alpha=0.2)
plt.title(f'Courbe de Lorenz - Concentration du CA B2B\nIndice de Gini : {gini:.2f}')
plt.xlabel('Part cumulée des clients')
plt.ylabel('Part cumulée du CA')
plt.legend()
plt.grid(False)
plt.show()

Le fait que l'indice grimpe à 0.44 quand on mélange tous les clients est logique : l'écart entre le plus petit client B2C (qui achète un livre à 5€) et le plus gros client B2B (qui achète pour 250 000€) est énorme.
Forte concentration due à la présence des 4 clients "Outliers".

In [ ]:
df_b2c.to_csv(r'..\..\projet9_Lapage\data\processed\df_b2c.csv',sep=';', encoding='utf-8', index=False)